# 第 4 章互动实验：透视扭曲与矫正

本实验生成网格并调用 OpenCV 作为几何基线，不包含 `4-1_齐次点变换.py` 的答案。先拖动透视强度，观察有效像素、插值误差和点顺序；后面三个实验对应教程 4.1、4.3 节的实测，其中实验 2、3 要用到你自己写的 `transform_points`。

In [ ]:
%matplotlib inline
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Heiti SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

height, width = 180, 240
grid = np.full((height, width), 245, dtype=np.uint8)
for x in range(0, width, 20): cv2.line(grid, (x, 0), (x, height-1), 80, 1)
for y in range(0, height, 20): cv2.line(grid, (0, y), (width-1, y), 80, 1)
cv2.putText(grid, 'TOP', (85, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, 0, 2)
src = np.float32([[0, 0], [width-1, 0], [width-1, height-1], [0, height-1]])

In [ ]:
def show_perspective(tilt=35):
    dst = np.float32([[tilt, 8], [width-1-tilt//2, 0], [width-1, height-1], [0, height-1-tilt//3]])
    H = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(grid, H, (width, height), flags=cv2.INTER_LINEAR, borderValue=255)
    H_inv = cv2.getPerspectiveTransform(dst, src)
    restored = cv2.warpPerspective(warped, H_inv, (width, height), flags=cv2.INTER_LINEAR, borderValue=255)
    mae = float(np.mean(np.abs(restored.astype(np.float32) - grid.astype(np.float32))))
    projected = cv2.perspectiveTransform(src.reshape(-1, 1, 2), H).reshape(-1, 2)
    corner_error = float(np.max(np.linalg.norm(projected - dst, axis=1)))
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, image, title in zip(axes, [grid, warped, restored], ['原网格', '透视扭曲', f'反变换：MAE={mae:.2f}']):
        ax.imshow(image, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title); ax.axis('off')
    plt.suptitle(f'四角重投影最大误差 {corner_error:.2e} 像素')
    plt.tight_layout(); plt.show()

interact(show_perspective, tilt=IntSlider(min=0, max=90, step=5, value=35));

## 接上自己的作业

下面的实验有一部分要调用你在 `4-1_齐次点变换.py` 里写的 `transform_points`。先运行这个单元把作业文件加载进来。作业还没写完也可以往下运行，用到它的单元会打印一行提示，不会报错；写完后回来重新运行这个单元。

每个实验都是先写下预测，再运行，然后把结果和原因记到 `实验记录.md`。

In [ ]:
import importlib.util, os
from pathlib import Path

CHAPTER_NAME = "04_几何变换"
ASSIGNMENT = "4-1_齐次点变换.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

def load_assignment():
    """加载作业文件；设置了环境变量 COURSE_IMPL_DIR 时改用那里的实现（维护者验证用）。"""
    chapter = find_chapter(CHAPTER_NAME)
    override = os.environ.get("COURSE_IMPL_DIR")
    base = chapter
    if override:
        # 相对路径先按当前目录找，找不到再按课程根目录找
        roots = [Path(override), chapter.parent / override]
        base = next((r / chapter.name for r in roots if (r / chapter.name / ASSIGNMENT).is_file()), roots[0] / chapter.name)
    spec = importlib.util.spec_from_file_location(Path(ASSIGNMENT).stem.replace("-", "_"), base / ASSIGNMENT)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，不中断 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

mod = load_assignment()   # 改了作业文件之后，重新运行这个单元再往下做
print("作业文件：", find_chapter(CHAPTER_NAME) / ASSIGNMENT)


## 实验 1：三种插值各往返一次

网格图的线宽只有 1 px，没有任何模糊。把它做一次透视变换再变回来，和原图比。

先预测：最近邻、双线性、双三次，哪一种的平均绝对误差最小？`tilt` 从 35 加到 90，排序会不会变？

运行后对照教程 4.1 节“插值误差会累积”的第 5 条，再想一想：换成一张相机拍的、带一点模糊的图，排序还会一样吗（同一节的往返实验给了答案）。

In [ ]:
def round_trip(tilt, flag):
    dst = np.float32([[tilt, 8], [width-1-tilt//2, 0], [width-1, height-1], [0, height-1-tilt//3]])
    warped = cv2.warpPerspective(grid, cv2.getPerspectiveTransform(src, dst), (width, height), flags=flag, borderValue=255)
    restored = cv2.warpPerspective(warped, cv2.getPerspectiveTransform(dst, src), (width, height), flags=flag, borderValue=255)
    return float(np.mean(np.abs(restored.astype(np.float32) - grid.astype(np.float32)))), restored

methods = [('最近邻', cv2.INTER_NEAREST), ('双线性', cv2.INTER_LINEAR), ('双三次', cv2.INTER_CUBIC)]
print('透视变换过去再变回来的平均绝对误差（灰度级）')
for tilt in (35, 90):
    print(f'tilt={tilt}：' + '，'.join(f'{name} {round_trip(tilt, flag)[0]:.2f}' for name, flag in methods))

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
axes[0].imshow(grid[10:70, 70:150], cmap='gray', vmin=0, vmax=255, interpolation='nearest')
axes[0].set_title('原图（局部）'); axes[0].axis('off')
for ax, (name, flag) in zip(axes[1:], methods):
    mae, restored = round_trip(35, flag)
    ax.imshow(restored[10:70, 70:150], cmap='gray', vmin=0, vmax=255, interpolation='nearest')
    ax.set_title(f'{name}：MAE={mae:.2f}'); ax.axis('off')
plt.suptitle('tilt=35 往返一次之后的同一块局部'); plt.tight_layout(); plt.show()

## 实验 2：第三个分量过零、整体缩放、相乘的顺序（用到作业 4-1）

先预测三件事：

1. 作业文件里演示用的矩阵作用在直线 y=0 上相邻的三个点 (-99, 0)、(-100, 0)、(-101, 0) 上，三个输出各在哪里？
2. 把矩阵整体乘 7，再整体乘 -1，输出变不变？
3. 点 (50, 20) 先绕原点转 30 度、再沿 x 平移 100，和先平移、再旋转，两个落点相差多远？

运行后对照教程 4.3 节。`cv2.perspectiveTransform` 对第三个分量为 0 的点返回什么，也顺便看一眼。

In [ ]:
H_demo = np.array([[1, 0.2, 3], [0.1, 1, 2], [0.01, 0.02, 1]], dtype=np.float64)
line = np.array([[-99.0, 0.0], [-100.0, 0.0], [-101.0, 0.0]])
mapped = attempt(mod.transform_points, line, H_demo)
if mapped is not None:
    w = line @ H_demo[2, :2] + H_demo[2, 2]          # 第三个分量 w = h31*x + h32*y + h33
    by_opencv = cv2.perspectiveTransform(line.reshape(-1, 1, 2), H_demo).reshape(-1, 2)
    for p, wi, q, c in zip(line, w, mapped, by_opencv):
        print(f'输入 {p}，w = {wi:+.2f}，你的输出 {np.round(q, 1)}，OpenCV 的输出 {np.round(c, 1)}')
    for factor in (7.0, -1.0):
        scaled = mod.transform_points(line, factor * H_demo)
        same = np.allclose(scaled, mapped, equal_nan=True)
        print(f'矩阵整体乘 {factor:g} 之后输出' + ('不变' if same else '变了，检查除法和无效点的判断'))

    theta = np.deg2rad(30)
    R = np.array([[np.cos(theta), -np.sin(theta), 0], [np.sin(theta), np.cos(theta), 0], [0, 0, 1]])
    T = np.array([[1, 0, 100], [0, 1, 0], [0, 0, 1]], dtype=np.float64)
    point = np.array([[50.0, 20.0]])
    rotate_first = mod.transform_points(point, T @ R)[0]
    shift_first = mod.transform_points(point, R @ T)[0]
    print(f'T @ R（先转后移）：{np.round(rotate_first, 1)}；R @ T（先移后转）：{np.round(shift_first, 1)}；'
          f'相距 {np.linalg.norm(rotate_first - shift_first):.1f} px')

## 实验 3：点序错了会怎样（用到作业 4-1）

把 `dst` 的前两个点对调，相当于把“左上”和“右上”标反了。

先预测：`getPerspectiveTransform` 会报错吗？变换后的图是什么样？原图四个角的第三个分量 w 还是同一个符号吗？

运行后对照教程 4.3 节“第三个分量过零时会发生什么”和 4.9 节表格里“透视变换的结果被撕开”一行。

In [ ]:
tilt = 35
dst = np.float32([[tilt, 8], [width-1-tilt//2, 0], [width-1, height-1], [0, height-1-tilt//3]])
swapped = dst[[1, 0, 2, 3]]                       # 前两个点对调
cases = [('点序正确', cv2.getPerspectiveTransform(src, dst)), ('前两个点对调', cv2.getPerspectiveTransform(src, swapped))]

# 沿原图边框取一圈点，后面用你的 transform_points 映射过去
t = np.linspace(0, 1, 40, endpoint=False)[:, None]
border = np.vstack([src[i] + t * (src[(i + 1) % 4] - src[i]) for i in range(4)]).astype(np.float64)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, H) in zip(axes, cases):
    corner_w = src.astype(np.float64) @ H[2, :2] + H[2, 2]
    print(f'{name}：四个角的 w = {np.round(corner_w, 2)}，' + ('同号' if (corner_w > 0).all() or (corner_w < 0).all() else '有正有负，w=0 的那条线穿过了图像'))
    ax.imshow(cv2.warpPerspective(grid, H, (width, height), borderValue=255), cmap='gray', vmin=0, vmax=255)
    ax.set_title(name); ax.axis('off')

outline = attempt(mod.transform_points, border, cases[1][1])
axes[2].set_title('边框上的点经过“对调”的矩阵')
if outline is not None:
    far = int((np.abs(outline) > 1000).any(axis=1).sum())
    print(f'边框上 {len(border)} 个点里，有 {far} 个被映射到 1000 px 以外，{int(np.isnan(outline).any(axis=1).sum())} 个是 NaN')
    axes[2].scatter(outline[:, 0], outline[:, 1], c=np.arange(len(border)), cmap='viridis', s=14)
    axes[2].plot([0, width-1, width-1, 0, 0], [0, 0, height-1, height-1, 0], color='gray', linewidth=1)
    axes[2].set_xlim(-400, 640); axes[2].set_ylim(580, -400); axes[2].set_aspect('equal')
    axes[2].set_xlabel('颜色从深到浅是沿边框走一圈的顺序，灰框是画布')
else:
    axes[2].text(0.5, 0.5, '完成作业后重新运行', ha='center', va='center', transform=axes[2].transAxes); axes[2].axis('off')
plt.tight_layout(); plt.show()

## 破坏实验与记录

1. 在最上面的互动单元里把 `tilt` 提高，观察反变换虽能运行但插值误差如何上升；再和实验 1 的三种插值对照。
2. 实验 3 里把对调的点换成别的组合（例如 `dst[[0, 1, 3, 2]]`、`dst[[2, 1, 0, 3]]`），先预测哪一种会让 w 变号，哪一种只是把图翻过来。
3. 思考为什么四角重投影误差很小，恢复图像仍不等于原图。
4. 实验 2 里把三个点换成 (-99.99, 0)、(-100, 0)、(-100.01, 0)，输出会跑到多远？

把矩阵、点顺序、MAE 和失败原因写入 `实验记录.md`。